In [5]:
### Using mask 1
import os
import time
import arcpy
from arcpy.sa import Raster, ExtractByMask

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------
root = r"F:\Topodata_tiles\Norway Derivs"
template_mask = r"F:\Topodata_basins\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"  # snap + mask + template grid

out_root = r"F:\Topodata_tiles\Test_no_clip_mosaics"
tmp_root = r"F:\Topodata_tiles\_tmp_mosaics"

os.makedirs(out_root, exist_ok=True)
os.makedirs(tmp_root, exist_ok=True)

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# ------------------------------------------------------------------
# OPTIONAL: run only selected variables for testing
# ------------------------------------------------------------------
ONLY_RUN = None  # e.g. {"Aspect(cos)"} or {"VD1", "Hillshade (MD)"}

# ------------------------------------------------------------------
# TEMPLATE GRID CONTROL (DTM MASK)
# ------------------------------------------------------------------
tmpl_desc = arcpy.Describe(template_mask)
tmpl_sr = tmpl_desc.spatialReference
tmpl_ext = tmpl_desc.extent
tmpl_rect = f"{tmpl_ext.XMin} {tmpl_ext.YMin} {tmpl_ext.XMax} {tmpl_ext.YMax}"

tmpl_r = arcpy.Raster(template_mask)
tmpl_w, tmpl_h = tmpl_r.width, tmpl_r.height
tmpl_cx, tmpl_cy = tmpl_r.meanCellWidth, tmpl_r.meanCellHeight

arcpy.env.snapRaster = template_mask
arcpy.env.extent = tmpl_ext
arcpy.env.cellSize = 10
arcpy.env.outputCoordinateSystem = tmpl_sr
arcpy.env.compression = "LZW"
arcpy.env.pyramid = "NONE"
arcpy.env.rasterStatistics = "NONE"

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def safe_name(folder_name: str) -> str:
    return (folder_name.replace(" ", "_")
                      .replace("/", "_")
                      .replace("\\", "_")
                      .replace(":", "_"))

def find_rasters(folder: str):
    """ArcGIS-safe raster listing (ignores .ovr/.html/etc.)."""
    old_ws = arcpy.env.workspace
    try:
        arcpy.env.workspace = folder
        rasters = arcpy.ListRasters() or []
        return [os.path.join(folder, r) for r in rasters]
    finally:
        arcpy.env.workspace = old_ws

def choose_resampling(folder_name: str) -> str:
    # continuous derivatives -> bilinear
    return "BILINEAR"

def fmt_seconds(s: float) -> str:
    s = max(0, int(round(s)))
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    if h > 0:
        return f"{h:d}:{m:02d}:{sec:02d}"
    return f"{m:d}:{sec:02d}"

def console_bar(i: int, n: int, width: int = 28) -> str:
    if n <= 0:
        return "[????????????????????????????]"
    frac = min(1.0, max(0.0, i / n))
    filled = int(round(frac * width))
    return "[" + "█" * filled + "·" * (width - filled) + "]"

def check_template_match(out_path: str, tmpl_raster: arcpy.Raster) -> bool:
    """Check rows/cols/cellsize/extent match template exactly."""
    out_r = arcpy.Raster(out_path)

    same_dims = (out_r.width == tmpl_raster.width) and (out_r.height == tmpl_raster.height)
    same_cell = (abs(out_r.meanCellWidth - tmpl_raster.meanCellWidth) < 1e-9) and (abs(out_r.meanCellHeight - tmpl_raster.meanCellHeight) < 1e-9)

    te = tmpl_raster.extent
    oe = out_r.extent
    same_ext = (abs(te.XMin - oe.XMin) < 1e-6 and abs(te.YMin - oe.YMin) < 1e-6 and
                abs(te.XMax - oe.XMax) < 1e-6 and abs(te.YMax - oe.YMax) < 1e-6)

    ok = same_dims and same_cell and same_ext

    msg = (f"    CHECK grid: dims({out_r.width}x{out_r.height}) "
           f"cell({out_r.meanCellWidth},{out_r.meanCellHeight}) "
           f"extent_match={same_ext} -> {'OK' if ok else 'WARNING'}")

    print(msg)
    arcpy.AddMessage(msg)

    if not ok:
        warn = f"    TEMPLATE MISMATCH! Template dims {tmpl_raster.width}x{tmpl_raster.height}, output dims {out_r.width}x{out_r.height}"
        print(warn)
        arcpy.AddWarning(warn)

    return ok

# ------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------
subfolders = [f for f in sorted(os.listdir(root)) if os.path.isdir(os.path.join(root, f))]
if ONLY_RUN is not None:
    subfolders = [f for f in subfolders if f in ONLY_RUN]

total = len(subfolders)
if total == 0:
    raise RuntimeError(f"No matching subfolders found in: {root} (check ONLY_RUN and folder names)")

mask_r = Raster(template_mask)

# ArcGIS Pro progress bar (geoprocessing UI)
arcpy.SetProgressor("step", "Mosaicking + aligning + masking to DTM template...", 0, total, 1)

global_start = time.time()
times = []

print(f"Found {total} derivative folders to process.")

for idx, folder_name in enumerate(subfolders, start=1):
    folder_start = time.time()

    folder_path = os.path.join(root, folder_name)
    rasters = find_rasters(folder_path)

    # Update ArcGIS progress label
    arcpy.SetProgressorLabel(f"({idx}/{total}) {folder_name}")

    if not rasters:
        msg = f"[SKIP] {folder_name}: no rasters found"
        print(msg)
        arcpy.AddMessage(msg)
        arcpy.SetProgressorPosition(idx)
        continue

    # ETA (based on completed folders)
    elapsed = time.time() - global_start
    if times:
        avg = sum(times) / len(times)
        eta = avg * (total - (idx - 1))
    else:
        eta = 0

    bar = console_bar(idx - 1, total)
    header = f"{bar} {idx-1}/{total}  elapsed {fmt_seconds(elapsed)}  ETA {fmt_seconds(eta)}"
    print("\n" + header)
    print(f"[PROCESS] {folder_name} | tiles: {len(rasters)}")
    arcpy.AddMessage(header)
    arcpy.AddMessage(f"[PROCESS] {folder_name} | tiles: {len(rasters)}")

    out_base = safe_name(folder_name)
    tmp_mosaic = os.path.join(tmp_root, f"{out_base}_mosaic.tif")
    tmp_proj = os.path.join(tmp_root, f"{out_base}_proj.tif")
    tmp_clip = os.path.join(tmp_root, f"{out_base}_clip.tif")
    out_final = os.path.join(out_root, f"{out_base}.tif")

    # Skip if already done
    if arcpy.Exists(out_final):
        msg = f"[SKIP] {folder_name}: output exists -> {out_final}"
        print(msg)
        arcpy.AddMessage(msg)
        arcpy.SetProgressorPosition(idx)
        continue

    try:
        # Bands from first tile
        r0 = arcpy.Raster(rasters[0])
        bands = r0.bandCount

        # 1) Mosaic tiles
        arcpy.management.MosaicToNewRaster(
            input_rasters=rasters,
            output_location=tmp_root,
            raster_dataset_name_with_extension=os.path.basename(tmp_mosaic),
            coordinate_system_for_the_raster=tmpl_sr,
            pixel_type="32_BIT_FLOAT",
            cellsize="",
            number_of_bands=bands,
            mosaic_method="LAST",
            mosaic_colormap_mode="FIRST"
        )

        # 2) Project/resample to template SR and 10 m (snap env ensures alignment)
        resampling = choose_resampling(folder_name)
        arcpy.management.ProjectRaster(
            in_raster=tmp_mosaic,
            out_raster=tmp_proj,
            out_coor_system=tmpl_sr,
            resampling_type=resampling,
            cell_size=10
        )

        # 3) Hard clip to template extent (prevents off-by-1)
        arcpy.management.Clip(
            in_raster=tmp_proj,
            rectangle=tmpl_rect,
            out_raster=tmp_clip,
            in_template_dataset="#",
            nodata_value="#",
            clipping_geometry="NONE",
            maintain_clipping_extent="MAINTAIN_EXTENT"
        )

        # 4) Apply DTM mask (keep only where mask has data)
        out_masked = ExtractByMask(Raster(tmp_clip), mask_r)
        out_masked.save(out_final)

        # 5) Verify grid match
        check_template_match(out_final, tmpl_r)

        # Timing
        folder_time = time.time() - folder_start
        times.append(folder_time)

        msg = f"  -> DONE: {out_final} | variable time {fmt_seconds(folder_time)}"
        print(msg)
        arcpy.AddMessage(msg)

    except Exception as e:
        folder_time = time.time() - folder_start
        times.append(folder_time)
        err = f"[ERROR] {folder_name} after {fmt_seconds(folder_time)}: {e}"
        print(err)
        arcpy.AddWarning(err)

    # Advance ArcGIS progress bar
    arcpy.SetProgressorPosition(idx)

# Finish
total_elapsed = time.time() - global_start
final_bar = console_bar(total, total)
print(f"\n{final_bar} {total}/{total}  total elapsed {fmt_seconds(total_elapsed)}")
arcpy.ResetProgressor()
arcpy.AddMessage(f"All done. Total elapsed: {fmt_seconds(total_elapsed)}")

arcpy.CheckInExtension("Spatial")


Found 1 derivative folders to process.

[····························] 0/1  elapsed 0:00  ETA 0:00
[PROCESS] Aspect(sin) | tiles: 254
    CHECK grid: dims(119486x149587) cell(10.0,10.0) extent_match=True -> OK
  -> DONE: E:\Topodata_tiles\Test_no_clip_mosaics\Aspect(sin).tif | variable time 59:55

[████████████████████████████] 1/1  total elapsed 59:55


'CheckedOut'

In [3]:
#####Extracting by mask + ids maps with bad pixels as _ch

import os
import time
import arcpy
from arcpy.sa import Raster, SetNull, IsNull

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------
root = r"F:\Topodata_tiles\Test_no_clip"
template_mask = r"F:\Topodata_basins\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"

out_root = r"F:\Topodata_tiles\Test_no_clip_mosaics"
tmp_root = r"F:\Topodata_tiles\_tmp_mosaics"

os.makedirs(out_root, exist_ok=True)
os.makedirs(tmp_root, exist_ok=True)

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

ONLY_RUN = None  # None = all

BAD_PIXEL_ABS = 1e15

# ------------------------------------------------------------------
# TEMPLATE GRID CONTROL
# ------------------------------------------------------------------
tmpl_desc = arcpy.Describe(template_mask)
tmpl_sr = tmpl_desc.spatialReference
tmpl_ext = tmpl_desc.extent
tmpl_rect = f"{tmpl_ext.XMin} {tmpl_ext.YMin} {tmpl_ext.XMax} {tmpl_ext.YMax}"
tmpl_r = arcpy.Raster(template_mask)

arcpy.env.snapRaster = template_mask
arcpy.env.extent = tmpl_ext
arcpy.env.cellSize = 10
arcpy.env.outputCoordinateSystem = tmpl_sr
arcpy.env.compression = "LZW"
arcpy.env.pyramid = "NONE"
arcpy.env.rasterStatistics = "NONE"

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def safe_name(folder_name: str) -> str:
    return (folder_name.replace(" ", "_")
                      .replace("/", "_")
                      .replace("\\", "_")
                      .replace(":", "_"))

def find_rasters(folder: str):
    old_ws = arcpy.env.workspace
    try:
        arcpy.env.workspace = folder
        rasters = arcpy.ListRasters() or []
        return [os.path.join(folder, r) for r in rasters]
    finally:
        arcpy.env.workspace = old_ws

def choose_resampling(_folder_name: str) -> str:
    return "BILINEAR"

def fmt_seconds(s: float) -> str:
    s = max(0, int(round(s)))
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h:d}:{m:02d}:{sec:02d}" if h > 0 else f"{m:d}:{sec:02d}"

def console_bar(i: int, n: int, width: int = 28) -> str:
    if n <= 0:
        return "[????????????????????????????]"
    frac = min(1.0, max(0.0, i / n))
    filled = int(round(frac * width))
    return "[" + "█" * filled + "·" * (width - filled) + "]"

def check_template_match(out_path: str, tmpl_raster: arcpy.Raster) -> bool:
    out_r = arcpy.Raster(out_path)
    ok = (out_r.width == tmpl_raster.width and out_r.height == tmpl_raster.height and
          abs(out_r.meanCellWidth - tmpl_raster.meanCellWidth) < 1e-9 and
          abs(out_r.meanCellHeight - tmpl_raster.meanCellHeight) < 1e-9 and
          abs(out_r.extent.XMin - tmpl_raster.extent.XMin) < 1e-6 and
          abs(out_r.extent.YMin - tmpl_raster.extent.YMin) < 1e-6 and
          abs(out_r.extent.XMax - tmpl_raster.extent.XMax) < 1e-6 and
          abs(out_r.extent.YMax - tmpl_raster.extent.YMax) < 1e-6)
    msg = f"    CHECK grid -> {'OK' if ok else 'WARNING'} (dims {out_r.width}x{out_r.height})"
    print(msg); arcpy.AddMessage(msg)
    if not ok:
        arcpy.AddWarning("    TEMPLATE MISMATCH! rows/cols/cellsize/extent differ from template mask.")
    return ok

def raster_has_bad_pixels_quick(r_path: str, abs_thresh: float) -> bool:
    """Quick check using MIN/MAX. If it errors, assume bad (to be safe)."""
    try:
        mx = float(arcpy.management.GetRasterProperties(r_path, "MAXIMUM").getOutput(0))
        mn = float(arcpy.management.GetRasterProperties(r_path, "MINIMUM").getOutput(0))
        return (mx > abs_thresh) or (mn < -abs_thresh)
    except Exception:
        return True

def delete_if_exists(p):
    try:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)
    except Exception:
        pass

# ------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------
subfolders = [f for f in sorted(os.listdir(root)) if os.path.isdir(os.path.join(root, f))]
if ONLY_RUN is not None:
    # allow ONLY_RUN to be set as {"Aspect(sin)"} etc.
    if isinstance(ONLY_RUN, str):
        ONLY_RUN_SET = {ONLY_RUN}
    else:
        ONLY_RUN_SET = set(ONLY_RUN)
    subfolders = [f for f in subfolders if f in ONLY_RUN_SET]

total = len(subfolders)
if total == 0:
    raise RuntimeError(f"No matching subfolders found in: {root} (check ONLY_RUN and folder names)")

mask_r = Raster(template_mask)

arcpy.SetProgressor("step", "Mosaicking + aligning + masking to DTM template...", 0, total, 1)

global_start = time.time()
times = []

print(f"Found {total} derivative folders to process.")

for idx, folder_name in enumerate(subfolders, start=1):
    folder_start = time.time()
    folder_path = os.path.join(root, folder_name)
    rasters = find_rasters(folder_path)

    arcpy.SetProgressorLabel(f"({idx}/{total}) {folder_name}")

    if not rasters:
        msg = f"[SKIP] {folder_name}: no rasters found"
        print(msg); arcpy.AddMessage(msg)
        arcpy.SetProgressorPosition(idx)
        continue

    elapsed = time.time() - global_start
    eta = (sum(times) / len(times) * (total - (idx - 1))) if times else 0

    print("\n" + f"{console_bar(idx-1, total)} {idx-1}/{total}  elapsed {fmt_seconds(elapsed)}  ETA {fmt_seconds(eta)}")
    print(f"[PROCESS] {folder_name} | tiles: {len(rasters)}")

    out_base = safe_name(folder_name)

    tmp_mosaic = os.path.join(tmp_root, f"{out_base}_mosaic.tif")
    tmp_proj   = os.path.join(tmp_root, f"{out_base}_proj.tif")
    tmp_clip   = os.path.join(tmp_root, f"{out_base}_clip.tif")

    # Decide output name later (_ch only if needed)
    out_ok = os.path.join(out_root, f"{out_base}.tif")
    out_ch = os.path.join(out_root, f"{out_base}_ch.tif")

    if arcpy.Exists(out_ok) or arcpy.Exists(out_ch):
        existing = out_ch if arcpy.Exists(out_ch) else out_ok
        msg = f"[SKIP] {folder_name}: output exists -> {existing}"
        print(msg); arcpy.AddMessage(msg)
        arcpy.SetProgressorPosition(idx)
        continue

    try:
        # 1) Mosaic
        r0 = arcpy.Raster(rasters[0])
        bands = r0.bandCount

        arcpy.management.MosaicToNewRaster(
            input_rasters=rasters,
            output_location=tmp_root,
            raster_dataset_name_with_extension=os.path.basename(tmp_mosaic),
            coordinate_system_for_the_raster=tmpl_sr,
            pixel_type="32_BIT_FLOAT",
            cellsize="",
            number_of_bands=bands,
            mosaic_method="LAST",
            mosaic_colormap_mode="FIRST"
        )

        # 2) Project
        arcpy.management.ProjectRaster(
            in_raster=tmp_mosaic,
            out_raster=tmp_proj,
            out_coor_system=tmpl_sr,
            resampling_type=choose_resampling(folder_name),
            cell_size=10
        )

        # 3) Clip to template extent
        arcpy.management.Clip(
            in_raster=tmp_proj,
            rectangle=tmpl_rect,
            out_raster=tmp_clip,
            in_template_dataset="#",
            nodata_value="#",
            clipping_geometry="NONE",
            maintain_clipping_extent="MAINTAIN_EXTENT"
        )

        # 4) Streamed mask + bad pixel clean in ONE expression, write only FINAL
        r_clip = Raster(tmp_clip)

        # quick check for bad pixels (based on properties)
        bad = raster_has_bad_pixels_quick(tmp_clip, BAD_PIXEL_ABS)
        out_final = out_ch if bad else out_ok
        if bad:
            arcpy.AddWarning(f"    BAD PIXELS suspected/detected (abs > {BAD_PIXEL_ABS:g}). Writing _ch output.")

        # Keep only where mask has data (IsNull(mask) => outside land => NoData)
        r_masked = SetNull(IsNull(mask_r), r_clip)

        # Remove extreme junk
        r_clean = SetNull((r_masked > BAD_PIXEL_ABS) | (r_masked < -BAD_PIXEL_ABS), r_masked)

        # Write final
        r_clean.save(out_final)

        # verify
        check_template_match(out_final, tmpl_r)

        dt = time.time() - folder_start
        times.append(dt)
        print(f"  -> DONE: {out_final} | variable time {fmt_seconds(dt)}")
        arcpy.AddMessage(f"  -> DONE: {out_final} | variable time {fmt_seconds(dt)}")

    except Exception as e:
        dt = time.time() - folder_start
        times.append(dt)
        err = f"[ERROR] {folder_name} after {fmt_seconds(dt)}: {e}"
        print(err); arcpy.AddWarning(err)

    finally:
        # CRITICAL: delete huge temps each variable to avoid disk filling
        delete_if_exists(tmp_clip)
        delete_if_exists(tmp_proj)
        delete_if_exists(tmp_mosaic)

    arcpy.SetProgressorPosition(idx)

total_elapsed = time.time() - global_start
print(f"\n{console_bar(total, total)} {total}/{total}  total elapsed {fmt_seconds(total_elapsed)}")
arcpy.ResetProgressor()
arcpy.AddMessage(f"All done. Total elapsed: {fmt_seconds(total_elapsed)}")

arcpy.CheckInExtension("Spatial")



Found 3 derivative folders to process.

[····························] 0/3  elapsed 0:02  ETA 0:00
[PROCESS] Aspect(cos) | tiles: 254
    CHECK grid -> OK (dims 119486x149587)
  -> DONE: F:\Topodata_tiles\Test_no_clip_mosaics\Aspect(cos)_ch.tif | variable time 1:43:15

[█████████···················] 1/3  elapsed 1:43:19  ETA 3:26:29
[PROCESS] Hillshade (MD) | tiles: 254
    CHECK grid -> OK (dims 119486x149587)
  -> DONE: F:\Topodata_tiles\Test_no_clip_mosaics\Hillshade_(MD)_ch.tif | variable time 1:07:47

[███████████████████·········] 2/3  elapsed 2:51:08  ETA 1:25:31
[PROCESS] VD1 | tiles: 254
    CHECK grid -> OK (dims 119486x149587)
  -> DONE: F:\Topodata_tiles\Test_no_clip_mosaics\VD1_ch.tif | variable time 1:03:26

[████████████████████████████] 3/3  total elapsed 3:54:35


'CheckedOut'